In [1]:
# PROBLEMI
# RISOLTO MA NON BEN IMPLEMENTATO 1. codice inefficiente/duplicato per polinomi pari
# RISOLTO IN V8 2. codice non funzionante/incompleto per funzioni generiche
# RISOLTO 3. problemi nel "circuito ultimo", andare dal prof (agg. 16/06/2026)

# versione ampliata della v7 ma senza codice completo per le funzioni generiche 
# e per lo switch con i polinomi sia pari che dispari. E' stato forzato l'inserimento di tanh(x))

In [2]:
import pennylane as qml
import numpy as np

# import pyqsp
# from pyqsp.angle_sequence import QuantumSignalProcessingPhases
from numpy.polynomial import Chebyshev, Polynomial

In [53]:
# ==========================================
# PARTE 1: I DATI E I REGISTRI/FILI
# ==========================================
# 1. Partiamo da una VERA matrice diagonale 4x4 (L'input)
# X_matrice = np.array([
#     [ 0.5,   0,   0,   0],
#     [  0, -0.2,   0,   0],
#     [  0,   0, 0.8,   0],
#     [  0,   0,   0,   0.1]
# ])

# X_matrice = np.array([
#     [ 0.7,   0,   0,   0],
#     [  0, -4,   0,   0],
#     [  0,   0, 8,   0],
#     [  0,   0,   0,   0.3]
# ])

# X_matrice = np.array([
#     [ -720,   0,   0,   0],
#     [  0, 56,   0,   0],
#     [  0,   0, 200,   0],
#     [  0,   0,   0,   170]
# ])

# X_matrice = np.array([
#     [ -720,   0],
#     [  0, 56]
# ])

# X_matrice = np.array([
#     [ -720,   0,   0],
#     [  0, 56,   0],
#     [  0,   0, 200],
# ])

#normalizzati gia

# # SET 1: Valori Piccoli (Zona lineare)
# X_matrice = np.array([
#     [ 0.15,    0,    0,    0],
#     [    0, -0.05,    0,    0],
#     [    0,    0,  0.22,    0],
#     [    0,    0,    0, -0.10]
# ])

# # SET 2: Valori Estremi (Saturazione forte)
# X_matrice = np.array([
#     [ 0.95,    0,    0,    0],
#     [    0, -0.88,    0,    0],
#     [    0,    0,  0.75,    0],
#     [    0,    0,    0, -0.99]
# ])

# # SET 3: Misto e Realistico
# X_matrice = np.array([
#     [ 0.60,    0,    0,    0],
#     [    0, -0.30,    0,    0],
#     [    0,    0, -0.85,    0],
#     [    0,    0,    0,  0.10]
# ])

# ==========================================
# TEST 16x16 (N = 16, address = 4 qubit, data = 4 qubit)
# ==========================================
X_matrice = np.diag([
     8.0,  -3.2,   0.5,  12.1, 
    -9.9,   4.4,  -0.1,   7.7,
    -5.5,   2.8, -11.0,   6.3, 
    -1.5,  10.2,  -4.8,   1.1
])

print("1. La matrice X di partenza è:")
print(X_matrice)
print("\n")


# 2. Estraiamo la diagonale classica
x_classico_grezzo = np.diag(X_matrice)
N_originale = len(x_classico_grezzo)

print("2. La diagonale di X e la sua dimensione:")
print("x_classico_grezzo =", x_classico_grezzo) 
print("N =", N_originale)
print("\n")

# 3. Calcolo qubit e Padding
# Calcoliamo n: il numero di qubit necessari (logaritmo in base 2 arrotondato per eccesso)
# Es: N=2 -> n=1. N=3 -> n=2. N=4 -> n=2.
n = int(np.ceil(np.log2(N_originale)))

# Calcoliamo quanti slot fisici avrà la memoria quantistica (la potenza di 2 più vicina)
N_pad = 2**n

# Facciamo il padding: aggiungiamo zeri alla fine se N_originale è minore di N_pad
# (Fondamentale per le matrici 3x3, 5x5, ecc.)
x_classico = np.pad(x_classico_grezzo, (0, N_pad - N_originale))

print("3. Calcolo qubit e Padding")
print("n =", n)
print("N_pad =", n)
print("x_classico =", x_classico)
print("\n")

# 4. Normalizzazione del vettore (obbligatoria per le ampiezze, sennò il computer quantistico non può leggere i dati)
norma_originale = np.linalg.norm(x_classico)
x = x_classico / norma_originale 

print("4. Normalizzazione di x_classico")
print("norma_originale =", norma_originale)
print("x =", x)

# 5. Assegniamo i fili. n=2 (perché abbiamo 4 dati, 2^2=4)
# Servono n fili per address, n per data, 1 per l'ancilla B, 1 per l'ancilla lcu.
# Generiamo le liste dei fili in base alla 'n' calcolata!
ancilla_finale = [0]                                    # Il nuovo qubit (+1) in alto
ancilla_pcorsivo = [1]
ancilla_gtilde = [2]
qubits_address = list(range(3, n+3))                   # Es: se n=1 -> [0]
qubits_dati = list(range(n+3, 2*n +3 ))                    # Es: se n=1 -> [1]
ancilla_b = [2*n +3]                                    # Es: se n=1 -> [2]

# Prepariamo un raggruppamento che ci servirà per S_0
qubits_da_B = qubits_dati + ancilla_b
qubits_ad_da_B = qubits_address + qubits_dati + ancilla_b
qubits_gtilde_ad_da_b = ancilla_gtilde + qubits_address + qubits_dati + ancilla_b
qubits_pcorsivo_gtilde_ad_da_b = ancilla_pcorsivo + ancilla_gtilde + qubits_address + qubits_dati + ancilla_b

# Creiamo una lista unica con tutti i 5 fili nell'ordine corretto
tutti_i_fili = ancilla_finale + ancilla_pcorsivo + ancilla_gtilde + qubits_address + qubits_dati + ancilla_b

# raggruppo le 2 ancille per la qsvt
ancille_qsvt = ancilla_pcorsivo + ancilla_gtilde

print(tutti_i_fili)

1. La matrice X di partenza è:
[[  8.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.   -3.2   0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.5   0.    0.    0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.   12.1   0.    0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.    0.   -9.9   0.    0.    0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.    0.    0.    4.4   0.    0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.    0.    0.    0.   -0.1   0.    0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.    0.    0.    0.    0.    7.7   0.    0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.    0.    0.    0.    0.    0.   -5.5   0.    0.    0.
    0.    0.    0.    0. ]
 [  0.    0.    0.    0.    0.    0.    0.    0.    0.    2.8   0.   

In [54]:
# ==========================================
# PARTE 2: ORACOLO E RIFLESSIONE
# ==========================================

def U_x(wires):
    """
    L'Oracolo: Inietta il vettore x nelle ampiezze dei qubit bersaglio.
    """
     #qml.AmplitudeEmbedding(features=x, wires=wires, normalize=True)
    qml.MottonenStatePreparation(x, wires=wires)

def S_0():
    """
    La Riflessione: La matrice diagonale con un -1 in alto a sinistra.
    Inverte il segno SOLO se i qubit 'dati' e 'B' sono tutti a zero.
    Non tocca gli 'address' per non rovinare le coordinate!
    """
    stato_zero = np.zeros(len(qubits_da_B), dtype=int)
    qml.FlipSign(stato_zero, wires=qubits_da_B)



# ==========================================
# PARTE 3: W (dalla Figura 1)
# ==========================================

def W():
    """
    Allinea i dati x_k con le coordinate corrette della matrice diagonale.
    """
    # 1. U normale solo sui dati
    U_x(wires=qubits_dati)
    
    # 2. Hadamard sull'ancilla B
    qml.Hadamard(wires=ancilla_b)
    
    # 3. U inverso sui dati, controllato da B
    # qml.adjoint ribalta l'oracolo, qml.ctrl lo subordina a B
    qml.ctrl(qml.adjoint(U_x), control=ancilla_b)(wires=qubits_dati)
    
    # 4. Le porte Toffoli incrociate (da address a dati, controllate da B)
    # Servono a "saldare" il dato corretto all'indirizzo corretto
    for i in range(len(qubits_address)):
        # Toffoli prende: [controllo_1, controllo_2, bersaglio]
        qml.Toffoli(wires=[ancilla_b[0], qubits_address[i], qubits_dati[i]])
        
    # 5. Hadamard finale sull'ancilla B
    qml.Hadamard(wires=ancilla_b)



# ==========================================
# PARTE 4: G
# ==========================================

def G():
    # Costruisce l'operatore G = W S_0 W^\dagger Z_B.
    # Le operazioni si scrivono nell'ordine esatto in cui toccano i qubit.
    
    # 1. Z_B (Riflessione sull'ancilla)
    qml.PauliZ(wires=ancilla_b)
    
    # 2. W^\dagger (Smontaggio: PennyLane inverte tutto W in automatico)
    qml.adjoint(W)()
    
    # 3. S_0 (Riflessione sull'origine)
    S_0()
    
    # 4. W (Rimontaggio)
    W()



# ==========================================
# PARTE 5: Gtilde
# ==========================================

def G_tilde():
    """Implementa -1/2 (G + G_dagger) usando le porte esatte del paper"""
    
    # 1. Hadamard iniziale sull'ancilla LCU
    qml.Hadamard(wires=ancilla_gtilde)
    
    # 2. Sandwich PauliX per il controllo su 0 (Pallino bianco)
    qml.PauliX(wires=ancilla_gtilde)
    qml.ctrl(G, control=ancilla_gtilde)()  # G controllato standard
    qml.PauliX(wires=ancilla_gtilde)
    
    # 3. Controllo su 1 per G_dagger (Pallino nero)
    qml.ctrl(qml.adjoint(G), control=ancilla_gtilde)()
    
    # 4. Hadamard finale per ricongiungere gli stati
    qml.Hadamard(wires=ancilla_gtilde)
    
    # 5. La rotazione Rz(2pi) per ottenere il segno MENO globale
    qml.RZ(2 * np.pi, wires=ancilla_gtilde)


In [56]:
# ==========================================
# PARTE 6: QSVT (Il Programma Non Lineare)
# ==========================================

# --- GENERATORE UNIVERSALE ANGOLI ---
def genera_angoli_qsvt(funzione=None, coeff_diretti=None, grado_polinomio=3, tipo_parita='dispari'):
    """Calcola gli angoli di fase partendo da una f(x) o da un polinomio esatto"""
    print("\n--- GENERATORE DI ANGOLI QSVT ---")
    coefficienti = None
    
    # OPZIONE A: Polinomio diretto
    if coeff_diretti is not None:
        print("Modalità: Inserimento diretto del Polinomio")
        coefficienti = np.array(coeff_diretti, dtype=float)
        
    # OPZIONE B: Funzione generica f(x) (Approssimazione di Chebyshev)
    elif funzione is not None:
        print(f"Modalità: Approssimazione di Chebyshev di f(x) (Grado {grado_polinomio})")
        x_punti = np.linspace(-1, 1, 1000)
        y_punti = funzione(x_punti)
        cheb_approssimazione = Chebyshev.fit(x_punti, y_punti, deg=grado_polinomio)
        pol_standard = cheb_approssimazione.convert(kind=Polynomial)
        coefficienti = pol_standard.coef.copy()
        
    else:
        raise ValueError("Devi fornire una funzione o dei coefficienti!")

    # Pulizia Parità (obbligatoria per QSVT standard)
    for i in range(len(coefficienti)):
        if tipo_parita == 'dispari' and i % 2 == 0:
            coefficienti[i] = 0.0
        elif tipo_parita == 'pari' and i % 2 != 0:
            coefficienti[i] = 0.0

    # TAGLIO DEGLI ZERI (RISOLVE L'ERRORE DEL 6° GRADO!) <---
    # Rimuove gli zeri creati dal pulitore in coda all'array, così PyQSP non va in errore.
    coefficienti = np.trim_zeros(coefficienti, 'b')
    if len(coefficienti) == 0:
        coefficienti = np.array([0.0])

    # Controllo e normalizzazione del limite quantistico [-1, 1]
    x_test = np.linspace(-1, 1, 1000)
    max_valore = np.max(np.abs(np.polyval(coefficienti[::-1], x_test)))
    
    # SALVIAMO IL FATTORE DI SCALA <---
    fattore_di_scala = 1.0
    if max_valore > 0.999:
        fattore_di_scala = max_valore / 0.999
        # Rimpiccioliamo il polinomio e salviamo di quanto lo abbiamo ridotto
        coefficienti = coefficienti / fattore_di_scala
        
    # # Calcolo angoli PyQSP DEPRECATO DA ME STESSO
    # angoli = np.array(QuantumSignalProcessingPhases(coefficienti, signal_operator="Wx"))

#     test_poly_to_angles(coefficienti)
    
    # Calcolo angoli
    angoli = np.array(qml.poly_to_angles(coefficienti, "QSVT", angle_solver='iterative'))
        
    # RESTITUIAMO SIA GLI ANGOLI CHE LA SCALA <---
    return angoli, fattore_di_scala

# --- TEST POLY_TO_ANGLES ---
def test_poly_to_angles(poly):
    print("\n--- TEST DELLA FUNZIONE QML.POLY_TO_ANGLES ---")
    qsvt_angles = qml.poly_to_angles(poly, "QSVT")
    
    t = 0.2
    
    # Encode x in the top left of the matrix
    block_encoding = qml.RX(2 * np.arccos(t), wires=0)
    projectors = [qml.PCPhase(angle, dim=1, wires=0) for angle in qsvt_angles]
    
    @qml.qnode(qml.device("default.qubit"))
    def circuit_qsvt():
        qml.QSVT(block_encoding, projectors)
        return qml.state()
    
    output = qml.matrix(circuit_qsvt, wire_order=[0])()[0, 0]
    expected = sum(coef * (t**i) for i, coef in enumerate(poly))
    
    print("output qsvt: ", output.real)
    print("P(t) =       ", expected)
    print("\n")
    
# --- STAMPA POLINOMIO ---
def stampa_polinomio(coeff_ascendenti):
    """
    Prende una lista di coefficienti [x^0, x^1, x^2...] 
    e restituisce la stringa formattata (es: 0.1x^3 + 0.2x)
    """
    termini = []
    grado_max = len(coeff_ascendenti) - 1
    
    # Scorriamo la lista al contrario (dal grado più alto al più basso)
    for i, c in enumerate(reversed(coeff_ascendenti)):
        esponente = grado_max - i
        
        # Saltiamo i coefficienti a zero
        if c == 0:
            continue
            
        # Formattiamo in base all'esponente
        if esponente == 0:
            termini.append(f"{c}")
        elif esponente == 1:
            termini.append(f"{c}x")
        else:
            termini.append(f"{c}x^{esponente}")
            
    # Uniamo i pezzi e sistemiamo i segni negativi (evita "+ -")
    se_vuoto = "0" if not termini else " + ".join(termini).replace("+ -", "- ")
    return se_vuoto

# 1. Definiamo la nostra funzione o il nostro polinomio
print(f"\n--- IMPOSTAZIONE QSVT ---")

# --- SCEGLI QUI QUALE METODO USARE ---

# METODO A: Polinomio Diretto (Attivo)
# Inserisci qui i coefficienti dal grado PIÙ BASSO (x^0) al PIÙ ALTO.
# Puoi aggiungere quanti numeri vuoi!
# Esempio: 0.1x^3 + 0.4x^2 + 0.2x + 0.3

# lista_coefficienti = [0.2, 0.2, 1.6, 0.3]
# lista_coefficienti = [0, 0, 1, 0]
# lista_coefficienti = [0.3, 0.2, 0.4, 0.1]
# lista_coefficienti = [0.2, 0.2, 0.3, 0.1, 0.1, 0.05]
#lista_coefficienti = [0, 0.23, 0, 0.67, 0, 0.12, 0, 0.3] #dispari
#lista_coefficienti = [0, 1, 0, 1, 0, 1, 0, 1] #dispari
#lista_coefficienti = [0, 0.2, 0, 0.1, 0, 0.05, 0, 0.05] #dispari
#lista_coefficienti = [0.2, 0.2, 0.3, 0.1, 0.1, 0.05, 0.05] 
#lista_coefficienti = [0.2, 0, 0.3, 0, 0.1, 0, 0.05] #pari
#lista_coefficienti = [0.23, 0, 0.67, 0, 0.12, 0, 0.3] #pari
lista_coefficienti = [1, 0, 1, 0, 1, 0, 1] #pari
#lista_coefficienti = [0.2, 0, 0.1, 0, 0.05, 0, 0.05, 0, 0.12] #pari
#lista_coefficienti = [-1, 0, 2] #pari
# lista_coefficienti = [0, 1, 0, 0]
# lista_coefficienti = [0.1, 0.4, 0.2, 0.3]
# lista_coefficienti = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.4]
# lista_coefficienti = [0, 0, 0, 1]

# Se volessi fare un polinomio di 5° grado ti basterebbe allungarla:
# lista_coefficienti = [0.5, 0, -1.2, 0, 0, 0.8] # 0.8x^5 - 1.2x^2 + 0.5

print(f"Vogliamo applicare il polinomio: P(x) = {stampa_polinomio(lista_coefficienti)}")

# PER USARE POLINOMI PARI, DISPARIFICAZIONE
# Se il termine noto c'è (è diverso da 0), il polinomio è PARI. DEF ERRATA!!
isPari = False #all(lista_coefficienti[i] == 0 for i in range(1, len(lista_coefficienti), 2))


# METODO B: Funzione Generica (Decommenta per usare questo!)
def mia_funzione(x):
    return np.tanh(x)
print("Vogliamo applicare la funzione: f(x) = tanh(x)")

# 2. La Traduzione in Angoli di Fase (Phase Angles)
# In un ambiente di produzione reale, qui chiameresti un "angle solver" classico.
try:
    # Generiamo gli angoli veri con PyQSP!
    # Se usi il METODO B, cambia i parametri qui sotto in: funzione=mia_funzione
    # ---> RICEVIAMO IL FATTORE DI SCALA 'scala_qsvt' <---
#     if isPari:
#         print("Rilevato polinomio PARI")
#         angoli_fase_qsvt, scala_qsvt = genera_angoli_qsvt(coeff_diretti=lista_coefficienti, tipo_parita='pari')
#     else:
#         print("Rilevato polinomio DISPARI")
#         angoli_fase_qsvt, scala_qsvt = genera_angoli_qsvt(coeff_diretti=lista_coefficienti, tipo_parita='dispari')
    angoli_fase_qsvt, scala_qsvt = genera_angoli_qsvt(funzione=mia_funzione, tipo_parita='dispari')
except Exception as e:
    print(f"Errore  ({e}).")
    # Qui inseriamo 4 angoli fittizi (in radianti) per costruire fisicamente il circuito 
    # nel caso l'approssimazione fallisse numericamente.
    print("Fallback: uso angoli fittizi.")
    angoli_fase_qsvt = np.array([0.15, -0.42, 0.88, -0.15])
    # SCALA DI FALLBACK <---
    scala_qsvt = 1.0

# angoli_fase_qsvt = np.array([-np.pi/2, np.pi/2, 0.0])
# # angoli_fase_qsvt = np.array([0.5, -1.2, 3.4, -1.2, 0.5])
# scala_qsvt = 1.0 
    
print(f"Angoli di fase pre-calcolati per il circuito:\n{angoli_fase_qsvt}")
print("\n")

print(f"Scala qsvt:\n{scala_qsvt}")
print("\n")

# 3. PennyLane vuole un "Operatore" formale per la QSVT. Quindi calcoliamo la matrice 
# esatta del nostro G_tilde, estraiamo il blocco di nostro interesse e la convertiamo in un operatore unitario nativo.
matrice_G_tilde = qml.matrix(G_tilde, wire_order=qubits_gtilde_ad_da_b)()

# Assumiamo ancilla_lcu di indice 0
# Estraiamo blocco in alto a sinistra della matrice G_tilde
def get_G_tilde_block():   
    # The full matrix is size 2^n x 2^n. 
    # If ancilla is the first wire, the top-left block (ancilla=0) 
    # is the first half of the matrix rows and columns.
    dim = matrice_G_tilde.shape[0] // 2
    g_tilde_block = matrice_G_tilde[0:dim, 0:dim]
    
    return g_tilde_block 
matrice_G_tilde_block = get_G_tilde_block()

print("Matrice G_tilde 'ritagliata' =", np.linalg.eig(matrice_G_tilde_block)[0])
print("\n")

operatore_G_tilde = qml.QubitUnitary(matrice_G_tilde, wires=qubits_gtilde_ad_da_b)
    
# 4. Definiamo i "fili proiettore". 
# A noi serve langolino in alto a sinistra che corrisponde allo stato in cui 
# l'ancilla LCU è a ZERO. 
# La QSVT ha bisogno di saperlo per agire sul posto giusto!

# Definiamo i fili e lo stato (tutti zeri) dell'angolino in alto a sinistra
fili_proiettore = ancilla_pcorsivo+ancilla_gtilde
stato_zero = np.zeros(len(fili_proiettore), dtype=int)

# Creiamo una lista di Phase-Shift Controllers (uno per ogni angolo).
# 'dim=1' significa che la fase viene applicata ESATTAMENTE e SOLO allo stato 
# in cui i fili del proiettore sono a ZERO (il nostro angolino in alto a sinistra!).
lista_proiettori = [qml.PCPhase(angle, dim=1, wires=fili_proiettore) for angle in angoli_fase_qsvt]

print("\n--- COSTRUZIONE DEL GATE P_CORSIVO ---")

# 5. Definiamo formalmente il GATE P_corsivo
def P_corsivo(): #già circuito_qsvt
    """
    Il Gate P_corsivo: la trasformazione non lineare sui valori singolari.
    Applica la QSVT al block-encoding G_tilde.
    Prende: 1. Operatore Base | 2. Lista di [ProiettoreIn, ProiettoreOut] | 3. Angoli
    """
    qml.QSVT(operatore_G_tilde, lista_proiettori)
    
# 6. Test di Compilazione del Circuito
dev_test = qml.device("default.qubit", wires=tutti_i_fili)

@qml.qnode(dev_test)
def test_qsvt_build():
    # Prepariamo lo stato iniziale: Hadamard sugli Address per creare 
    # la sovrapposizione uniforme di tutti gli input.
    for wire in qubits_address:
        qml.Hadamard(wires=wire)
        
    # Applichiamo la scatola QSVT che incastra G_tilde + Polinomio
    P_corsivo()
    
    # Ritorniamo il vettore di stato finale
    return qml.state()

print("\n--- COMPILAZIONE CIRCUITO TEST QSVT P_CORSIVO ---")
try:
    stato_finale = test_qsvt_build()
    print("-> SUCCESSO: La compilazione del nodo QSVT su G_tilde è andata a buon fine!")
    print(f"-> Il vettore di stato risultante ha {len(stato_finale)} ampiezze complesse.")
except Exception as e:
    print(f"-> Errore nella compilazione QSVT: {e}")



#poly = np.array([0, 1.0, 0, -1/2, 0, 1/3])



--- IMPOSTAZIONE QSVT ---
Vogliamo applicare il polinomio: P(x) = 1x^6 + 1x^4 + 1x^2 + 1
Vogliamo applicare la funzione: f(x) = tanh(x)

--- GENERATORE DI ANGOLI QSVT ---
Modalità: Approssimazione di Chebyshev di f(x) (Grado 3)
Angoli di fase pre-calcolati per il circuito:
[-1.53542151  1.10914419  1.10914419  0.03537482]


Scala qsvt:
1.0


Matrice G_tilde 'ritagliata' = [ 0.29672603+1.57076883e-17j  0.29672603-1.36307129e-17j
  1.        +9.00507357e-17j -1.        -5.27355937e-16j
 -1.        +5.42101086e-17j -1.        +1.16899083e-16j
 -1.        -1.11022302e-16j  1.        +8.13659213e-17j
  1.        +1.66533454e-16j -1.        -2.98503399e-17j
  1.        +1.23525408e-16j -1.        -1.17671018e-16j
  1.        +6.90987611e-17j -1.        -5.05834639e-17j
  1.        +1.47727979e-16j -1.        -1.50081637e-16j
 -1.        -1.45203387e-16j  1.        -2.77555756e-17j
  1.        +1.24900090e-16j  1.        +1.83745163e-16j
 -1.        -1.66533454e-16j -1.        -1.01915004e-1

-> SUCCESSO: La compilazione del nodo QSVT su G_tilde è andata a buon fine!
-> Il vettore di stato risultante ha 4096 ampiezze complesse.


In [58]:
# ==========================================
# PARTE 7: CIRCUITO FINALE ED ESTRAZIONE (Fig. 4)
# ==========================================

print("\n--- ESECUZIONE CIRCUITO NTCA FINALE ---")

print("\n--- COSTRUZIONE DELLO STATO FINALE ---")

# 1. Costruiamo lo STATO quantistico
dev_finale = qml.device("default.qubit", wires=tutti_i_fili)

@qml.qnode(dev_finale)
def circuito_stato_finale():
    """
    H su address -> W -> P_corsivo -> W_dagger
    """
    
    # a) H sul registro 'add' (Inizializzazione della sovrapposizione)
    for wire in qubits_address:
        qml.Hadamard(wires=wire)

    qml.Hadamard(wires=ancilla_finale)
        
    # b) Gate W (Allinea i dati con gli indirizzi)
    qml.PauliX(wires=ancilla_finale)
    qml.ctrl(W, control=ancilla_finale)()
    
    # c) Gate P_corsivo (Applica il polinomio QSVT)
    qml.ctrl(P_corsivo, control=ancilla_finale)()
    
    # d) Gate W_dagger (Smonta l'allineamento per isolare i risultati puliti)
    qml.ctrl(qml.adjoint(W), control=ancilla_finale)()
    qml.PauliX(wires=ancilla_finale)
    
    qml.Hadamard(wires=ancilla_finale)
    
    # Restituisce il vettore di stato gigante da cui pescheremo i risultati
    return qml.state()

# @qml.qnode(dev_finale)
# def circuito_stato_finale():
#     """
#     H su address -> W -> P_corsivo -> W_dagger
#     """
    
#     # a) H sul registro 'add' (Inizializzazione della sovrapposizione)
#     for wire in qubits_address:
#         qml.Hadamard(wires=wire)
        
#     # b) Gate W (Allinea i dati con gli indirizzi)
#     W()
    
#     # c) Gate P_corsivo (Applica il polinomio QSVT)
#     P_corsivo()
    
#     # d) Gate W_dagger (Smonta l'allineamento per isolare i risultati puliti)
#     qml.adjoint(W)()
    
#     # Restituisce il vettore di stato gigante da cui pescheremo i risultati
#     return qml.state()

# 2. Esecuzione ed Estrazione
print("-> Esecuzione del circuito in corso...")
stato_ntca = circuito_stato_finale()
print("   Circuito eseguito! Vettore di stato ottenuto.\n")

print("--- RISULTATI ESTRATTI ---")
# lista inizializzata con 0, dove andranno i valori finali
risultati_quantistici = np.zeros(N_pad, dtype=complex)

# Il vettore ha 64 elementi. Cerchiamo la fetta in cui b_anc, data e B sono a 0.
for i, ampiezza in enumerate(stato_ntca):
    # Convertiamo l'indice in stringa binaria (es: '010110')
    binario = format(i, f'0{len(tutti_i_fili)}b')
    
    fin_str = binario[0]                   # 'b_anc' nel disegno
    pcorsivo_str = binario[1]
    gtilde_str = binario[2]
    addr_str = binario[3:3+n]              # 'add' nel disegno
    dati_e_b_str = binario[3+n:]           # 'data' e 'B' nel disegno
    
    # La regola: prendi solo se 'b_anc' è 0 E tutti i 'data' e 'B' sono 0
    if fin_str == '0' and pcorsivo_str == '0' and gtilde_str == '0' and all(bit == '0' for bit in dati_e_b_str):
        k = int(addr_str, 2) # Converte l'indirizzo da binario a decimale
        # Moltiplichiamo per sqrt(N_pad) per compensare le porte H iniziali
        risultati_quantistici[k] = ampiezza * np.sqrt(N_pad)

# Stampiamo i valori finali trovati negli address |k>
for k in range(N_pad):
    valore = risultati_quantistici[k]
    # Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
    valore_pulito = np.round(valore.real, 4) + 1j * np.round(valore.imag, 4)
    print(f"Indirizzo |{k}> : {valore_pulito}")

print("\n")

for k in range(N_pad):
    valore = risultati_quantistici[k]

    # 1. NELLA QSVT, IL RISULTATO È SOLO NELLA PARTE REALE
    valore_reale = valore.real

    # 2. Invertiamo matematicamente l'interferenza con l'Identità
    valore_reale_puro = (valore_reale * 2) - 1
    
    # 3. SE IL POLINOMIO ERA PARI, ANNULLIAMO LA DISPARIFICAZIONE
       # deprecato

    # 4. Denormalizziamo
    risultato_vero = valore_reale_puro * scala_qsvt
    
    # 5. Stampiamo il risultato finale pulito, Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
    print(f"Indirizzo |{k}> : {risultato_vero:.4f}")
    
print("\n")



--- ESECUZIONE CIRCUITO NTCA FINALE ---

--- COSTRUZIONE DELLO STATO FINALE ---
-> Esecuzione del circuito in corso...
   Circuito eseguito! Vettore di stato ottenuto.

--- RISULTATI ESTRATTI ---
Indirizzo |0> : (0.6424-0.3351j)
Indirizzo |1> : (0.4421+0.1481j)
Indirizzo |2> : (0.5091-0.0235j)
Indirizzo |3> : (0.7096-0.4255j)
Indirizzo |4> : (0.3258+0.3872j)
Indirizzo |5> : (0.5794-0.2003j)
Indirizzo |6> : (0.4982+0.0047j)
Indirizzo |7> : (0.6372-0.3255j)
Indirizzo |8> : (0.401+0.2455j)
Indirizzo |9> : (0.5507-0.1301j)
Indirizzo |10> : (0.3079+0.4096j)
Indirizzo |11> : (0.613-0.2764j)
Indirizzo |12> : (0.4728+0.0704j)
Indirizzo |13> : (0.6792-0.3939j)
Indirizzo |14> : (0.4134+0.2171j)
Indirizzo |15> : (0.52-0.0517j)


Indirizzo |0> : 0.2848
Indirizzo |1> : -0.1159
Indirizzo |2> : 0.0182
Indirizzo |3> : 0.4192
Indirizzo |4> : -0.3485
Indirizzo |5> : 0.1589
Indirizzo |6> : -0.0036
Indirizzo |7> : 0.2745
Indirizzo |8> : -0.1979
Indirizzo |9> : 0.1015
Indirizzo |10> : -0.3843
Indirizzo |1

In [33]:
def esegui_verifica_classica(dati_x, coeff_lista, polinomio_pari):
    print("--- VERIFICA DI COERENZA CLASSICA ---")
    
    for k, x_k in enumerate(dati_x):
        # Calcolo matematico puro: c0 + c1*x + c2*x^2 + ...
        risultato_puro = sum(c * (x_k ** esponente) for esponente, c in enumerate(coeff_lista))
        
        print(f"Indirizzo |{k}> : {risultato_puro:.4f}")
    print("\n")

# Chiamata alla funzione separata per il confronto
esegui_verifica_classica(dati_x=x, coeff_lista=lista_coefficienti, polinomio_pari=isPari)

--- VERIFICA DI COERENZA CLASSICA ---
Indirizzo |0> : 1.3622
Indirizzo |1> : 1.0309
Indirizzo |2> : 2.1126
Indirizzo |3> : 1.1360




In [59]:
# POSSIBILI PROBLEMI
# 1 U_2 incompleto
# 2 circuito_stato_finale non è dato nel modo corretto

#SOLO CON TANH(X) O 
if(isPari!=1):
    def circuito_stato_finale():
        """
        H su address -> W -> P_corsivo -> W_dagger
        """

        # a) H sul registro 'add' (Inizializzazione della sovrapposizione)
        for wire in qubits_address:
            qml.Hadamard(wires=wire)

        # b) Gate W (Allinea i dati con gli indirizzi)
        W()

        # c) Gate P_corsivo (Applica il polinomio QSVT)
        P_corsivo()

        # d) Gate W_dagger (Smonta l'allineamento per isolare i risultati puliti)
        qml.adjoint(W)()
        

    def U_2():
        """
        Questo U_2 agisce come il termine noto costante.
        Applica gli Hadamard SOLO agli address per distribuire la 
        costante equamente su tutti gli indirizzi (|0>, |1>, |2>, |3>).
        Lascia intatti i dati e le ancille (che quindi restano a 0,
        agendo come l'Identità perfetta sul registro dati).
        """
        for wire in qubits_address:
            qml.Hadamard(wires=wire)

    ancilla_ultima = [2*n +4]

    dev_ultimo = qml.device("default.qubit", wires=tutti_i_fili+ancilla_ultima)

    @qml.qnode(dev_ultimo)
    def circuito_dispari_termine_noto():
        """
        H su address -> W -> P_corsivo -> W_dagger
        """
        qml.Hadamard(wires=ancilla_ultima)
        qml.ctrl(circuito_stato_finale, control=ancilla_ultima)()
        qml.PauliX(wires=ancilla_ultima)
        qml.ctrl(U_2, control=ancilla_ultima)()
        qml.Hadamard(wires=ancilla_ultima)

        # Restituisce il vettore di stato gigante da cui pescheremo i risultati
        return qml.state()

    # 2. Esecuzione ed Estrazione
    print("-> Esecuzione del circuito in corso...")
    stato_ultimo = circuito_dispari_termine_noto()
    print("   Circuito eseguito! Vettore di stato ottenuto.\n")

    print("--- RISULTATI ESTRATTI ---")
    # lista inizializzata con 0, dove andranno i valori finali
    risultati_quantistici_u = np.zeros(N_pad, dtype=complex)

    # Il vettore ha 64 elementi. Cerchiamo la fetta in cui b_anc, data e B sono a 0.
    for i, ampiezza in enumerate(stato_ultimo):
        # Convertiamo l'indice in stringa binaria (es: '010110')
        binario = format(i, f'0{len(tutti_i_fili+ancilla_ultima)}b')
    
        fin_str = binario[0]                   # 'b_anc' nel disegno
        pcorsivo_str = binario[1]
        gtilde_str = binario[2]
        addr_str = binario[3:3+n]
        # -1 FERMA LA STRINGA ESATTAMENTE PRIMA DELL'ULTIMA ANCILLA
        dati_e_b_str = binario[3+n:-1]           # 'data' e 'B' nel disegno
        ultima = binario[-1]

        # La regola: prendi solo se 'b_anc' è 0 E tutti i 'data' e 'B' sono 0
        if ultima == '0' and fin_str == '0' and pcorsivo_str == '0' and gtilde_str == '0' and all(bit == '0' for bit in dati_e_b_str):
            k = int(addr_str, 2) # Converte l'indirizzo da binario a decimale
            # Moltiplichiamo per sqrt(N_pad) per compensare le porte H iniziali
            risultati_quantistici_u[k] = ampiezza * np.sqrt(N_pad)

    # Stampiamo i valori finali trovati negli address |k>
    for k in range(N_pad):
        valoreu = risultati_quantistici_u[k]
        # Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
        valore_pulito_u = np.round(valoreu.real, 4) + 1j * np.round(valoreu.imag, 4)
        print(f"Indirizzo |{k}> : {valore_pulito_u}")

    print("\n")

    for k in range(N_pad):
        valoreu = risultati_quantistici_u[k]

        # 1. NELLA QSVT, IL RISULTATO È SOLO NELLA PARTE REALE
        risultato_verou = valoreu.real*2

#         # 2. Invertiamo matematicamente l'interferenza con l'Identità
#         # Inversione matematica del DOPPIO LCU #errato
#         valore_reale_purou = (valore_realeu * 4) - 3

#         # 4. Denormalizziamo
#         risultato_verou = valore_reale_purou * scala_qsvt

        # 5. Stampiamo il risultato finale pulito, Arrotondiamo per pulire i piccolissimi errori di calcolo del simulatore
        print(f"Indirizzo |{k}> : {risultato_verou:.4f}")

    print("\n")



-> Esecuzione del circuito in corso...
   Circuito eseguito! Vettore di stato ottenuto.

--- RISULTATI ESTRATTI ---
Indirizzo |0> : (0.6424-0.3351j)
Indirizzo |1> : (0.4421+0.1481j)
Indirizzo |2> : (0.5091-0.0235j)
Indirizzo |3> : (0.7096-0.4255j)
Indirizzo |4> : (0.3258+0.3872j)
Indirizzo |5> : (0.5794-0.2003j)
Indirizzo |6> : (0.4982+0.0047j)
Indirizzo |7> : (0.6372-0.3255j)
Indirizzo |8> : (0.401+0.2455j)
Indirizzo |9> : (0.5507-0.1301j)
Indirizzo |10> : (0.3079+0.4096j)
Indirizzo |11> : (0.613-0.2764j)
Indirizzo |12> : (0.4728+0.0704j)
Indirizzo |13> : (0.6792-0.3939j)
Indirizzo |14> : (0.4134+0.2171j)
Indirizzo |15> : (0.52-0.0517j)


Indirizzo |0> : 1.2848
Indirizzo |1> : 0.8841
Indirizzo |2> : 1.0182
Indirizzo |3> : 1.4192
Indirizzo |4> : 0.6515
Indirizzo |5> : 1.1589
Indirizzo |6> : 0.9964
Indirizzo |7> : 1.2745
Indirizzo |8> : 0.8021
Indirizzo |9> : 1.1015
Indirizzo |10> : 0.6157
Indirizzo |11> : 1.2260
Indirizzo |12> : 0.9455
Indirizzo |13> : 1.3583
Indirizzo |14> : 0.8269
In